Четвертое домашние задание 

В качестве узлов дерева будет использоваться следующая структура:

In [ ]:
class Node():
    def __init__(self,val=0,left=None,right=None):
        self.value = val
        self.left = left
        self.right = right
        self.balance_factor=None

1. Полное бинарное дерево
Реализуйте функцию, которая проверяет, является ли заданное бинарное дерево
полным.
Необходимо корректно определить условие заполненности уровней и объяснить
выбранный способ обхода (например, BFS).

In [3]:
from collections import deque
def is_complete(root: Node|None) -> bool:
    if not root:
        return True # считаем пустое дерово полным
    deq=deque([root])
    is_None_was=False
    while deq:
        for _ in range(len(deq)):

            node=deq.popleft()
            if node is None:
                is_None_was=True
            else: 
                if is_None_was:
                    return False
                deq.append(node.left)
                deq.append(node.right)
    
    return True


BFS проходит все узлы в нужном нам порядке, потому что у полного дерева нет дырок и только нижний уровень может быть не заполнен, здесь если мы встретим дырку и что-то после нее в обходе в ширину функция выдаст False

2. Объединение K отсортированных массивов
Напишите функцию, объединяющую K отсортированных массивов в один
отсортированный массив.
Для эффективного решения используйте мин-кучу и проанализируйте сложность
алгоритма.

In [4]:
# напишем класс мин кучи
class MinHeap:
    def __init__(self):
        self.heap = []
    def parent(self, i):
        if i==0:
            return 0
        return (i-1)//2

    def left(self, i):
        return 2*i+1

    def right(self, i):
        return 2*i+2

    def sift_up(self, i):
        if i==0:
            return
        parent_idx=self.parent(i)
        if self.heap[parent_idx]>self.heap[i]:
            self.heap[parent_idx],self.heap[i]=self.heap[i],self.heap[parent_idx]
            self.sift_up(parent_idx)
        

    def sift_down(self, i):
        left_idx=self.left(i)
        right_idx=self.right(i)
        if left_idx>=len(self.heap):
            return
        elif left_idx<len(self.heap) and right_idx>=len(self.heap):
            if self.heap[i]>self.heap[left_idx]:
                self.heap[i],self.heap[left_idx]=self.heap[left_idx],self.heap[i]
                self.sift_down(left_idx)
            else:
                return
        else:
            min_val=min(self.heap[left_idx],self.heap[right_idx])
            if min_val==self.heap[left_idx]:
                candidate=left_idx
            else:
                candidate=right_idx
            if self.heap[i]>self.heap[candidate]:
                self.heap[i],self.heap[candidate]=self.heap[candidate],self.heap[i]
                self.sift_down(candidate)
            else: 
                return


    def push(self, value):
        self.heap.append(value)
        self.sift_up(len(self.heap)-1)

    def pop(self):
        if not self.heap:
            raise IndexError("pop from empty heap")

        self.heap[0],self.heap[-1]=self.heap[-1],self.heap[0]
        val=self.heap.pop()
        self.sift_down(0)
        return val

    def build_heap(self, nums: list[int]):
        self.heap = nums
        for i in range(len(self.heap)//2 - 1,-1,-1):
            self.sift_down(i)


Оказывается для этой задачи нужно еще хранить индексы массива, напишем для этого специализированную кучу...

In [5]:
class MinHeapWithIdx:
    def __init__(self):
        self.heap = []

    def parent(self, i):
        return (i - 1) // 2

    def left(self, i):
        return 2 * i + 1

    def right(self, i):
        return 2 * i + 2

    def sift_up(self, i):
        if i == 0:
            return

        parent_idx = self.parent(i)

        if self.heap[parent_idx][0] > self.heap[i][0]:
            self.heap[parent_idx], self.heap[i] = (
                self.heap[i], self.heap[parent_idx]
            )
            self.sift_up(parent_idx)

    def sift_down(self, i):
        left_idx = self.left(i)
        right_idx = self.right(i)

        if left_idx >= len(self.heap):
            return

        elif right_idx >= len(self.heap):
            if self.heap[i][0] > self.heap[left_idx][0]:
                self.heap[i], self.heap[left_idx] = (
                    self.heap[left_idx], self.heap[i]
                )
                self.sift_down(left_idx)

        else:
            if self.heap[left_idx][0] <= self.heap[right_idx][0]:
                candidate = left_idx
            else:
                candidate = right_idx

            if self.heap[i][0] > self.heap[candidate][0]:
                self.heap[i], self.heap[candidate] = (
                    self.heap[candidate], self.heap[i]
                )
                self.sift_down(candidate)

    def push(self, value, array_idx):
        self.heap.append((value, array_idx))
        self.sift_up(len(self.heap) - 1)

    def pop(self):
        if not self.heap:
            raise IndexError("pop from empty heap")

        self.heap[0], self.heap[-1] = (
            self.heap[-1], self.heap[0]
        )

        val = self.heap.pop()
        self.sift_down(0)
        return val

    def build_heap(self, nums: list[tuple[int, int]]):
        self.heap = nums.copy()

        for i in range(len(self.heap) // 2 - 1, -1, -1):
            self.sift_down(i)

In [7]:
# массивы отсортированны в порядке не возрастания, то есть [3,2,1,0]
def merge_k_sorted(arrays: list[list[int]]) -> list[int]:
    res=[]
    heap=MinHeapWithIdx()
    for i in range(len(arrays)):
        if arrays[i] !=[]:
            val=arrays[i].pop()
            heap.push(val,i)
    while heap.heap:
        val,idx=heap.pop()
        res.append(val)
        if arrays[idx]!=[]:
            val=arrays[idx].pop()
            heap.push(val,idx)

    return res


arrays = [
    [9, 6, 3, 1],
    [8, 5, 2],
    [10, 7, 4, 0]
]

print(merge_k_sorted(arrays))
    

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


K - число массивов
N - число элементов в них
В куче хранится не более одного элемента каждого массива, значит по памяти сложность O(K)
Мы проходим по всем K массивам и добавляем в мин-кучу последний элемент каждого непустого массива.
Одна операция push() занимает O(log K), поскольку размер кучи не превышает K
Следовательно, первоначальное заполнение занимает:
O(KlogK)
Каждый из N элементов ровно один раз извлекается из кучи.
На каждой итерации выполняются:
- pop() — O(log K)
- res.append() — O(1)
- arrays[idx].pop() — O(1), поскольку удаляется последний элемент списка.
- heap.push() — O(log K), если в исходном массиве остались элементы.
Таким образом, обработка всех элементов занимает: O(N log K)
Итоговая временная сложность: O(Nlog K)



3. K-ый наименьший / наибольший элемент
Дан массив чисел. Необходимо найти K-ый наименьший или K-ый наибольший
элемент.
Рассмотрите эффективные подходы (например, с использованием кучи) и обоснуйте
сложность решения.

Будем искать к ый наименьший потому что у меня мин куча
Нами достаточно построить мин кучу па массиву и  к раз сделать поп у мин кучи, тогда и получим ответ

In [10]:
def kth_smallest(nums:list[int], k:int)-> int:
    if k>len(nums):
        raise ValueError("k больше длинны массива")
    heap=MinHeap()
    heap.build_heap(nums)
    for _ in range(k-1):
        heap.pop()
    return heap.pop()

values = [1,2,3,4,5,6,7,8,9,0]

print(kth_smallest(values,3))


2


Если в массиве N элементов то построить кучу по нему это O(N)
Далее мы k раз попаем элемент и просеиваем вниз это O(Klog(N))
Итоговая сложность O(N + Klog(N))

4. K-ый наименьший / наибольший элемент в BST
Дано бинарное дерево поиска.
Реализуйте алгоритм нахождения K-ого наименьшего или наибольшего элемента,
используя свойства BST и обход дерева.

In [17]:
def kth_smallest_bst(root:Node|None, k:int)->int:
    result=None
    def dfs(root):
        nonlocal k,result
        if not root:
            return root
        dfs(root.left)
        k-=1
        if k==0:
            result=root.value
            return
        dfs(root.right)
    dfs(root)
    return result

# бинарное дерево
root = Node(8)

root.left = Node(3)
root.right = Node(10)

root.left.left = Node(1)
root.left.right = Node(6)

root.left.right.left = Node(4)
root.left.right.right = Node(7)

root.right.right = Node(14)
root.right.right.left = Node(13)
print(kth_smallest_bst(root,1))

1


5. Balance factor
На вход функции передаётся указатель на корень бинарного дерева поиска.
Необходимо для каждого узла вычислить и заполнить поле balance_factor (разницу
высот левого и правого поддеревьев), объяснив способ вычисления высоты.

In [18]:
def fill_balance_factors(root: Node | None) -> None:

    def dfs(node):
        if not node:
            return 0

        left_height = dfs(node.left)
        right_height = dfs(node.right)

        node.balance_factor = left_height - right_height

        return 1 + max(left_height, right_height)

    dfs(root)

Высота вычисляется рекурсивно с помощью DFS (Postorder). Сначала находим высоты левого и правого поддеревьев, затем высоту текущего узла по формуле h = 1 + max(h_left, h_right). Высота пустого дерева равна 0. Баланс-фактор вычисляется как разность высот левого и правого поддеревьев. Сложность алгоритма — O(N), так как каждый узел посещается один раз.

6. Преобразование в зеркальное дерево
Реализуйте алгоритм зеркального отражения бинарного дерева, меняя местами
левые и правые поддеревья каждого узла.
Поясните, почему алгоритм корректен и какова его временная сложность.

In [ ]:
def mirror_tree(root):
    if root is None:
        return root
    root.left, root.right = root.right, root.left
    mirror_tree(root.left)
    mirror_tree(root.right)
    return root

Алгоритм использует рекурсивный DFS. Для каждого узла меняем местами левое и правое поддеревья, после чего рекурсивно выполняем ту же операцию для потомков. Алгоритм корректен, поскольку зеркальное отражение всего дерева получается отражением каждого его поддерева. Временная сложность O(N), так как каждый узел посещается один раз. Дополнительная память O(H) для стека рекурсии, где H — высота дерева.